In [1]:
import pandas as pd
import os
import sys


In [2]:
# Adiciona o diretório-raiz do projeto ao sys.path
sys.path.append('/app')

In [ ]:
import importlib

import utils.conexao
importlib.reload(utils.conexao)
from utils.conexao import get_dados_origem, SCHEMA_ORIGEM

# consulta
query = f'SELECT * FROM "{SCHEMA_ORIGEM}"."competencia";'
df_competencia = get_dados_origem(query) 

if df_competencia is not None:
    print("Extração concluída:")
    print(df_competencia.head())

SyntaxError: invalid syntax (982538042.py, line 3)

In [ ]:
#mostrar 10 linhas
df_competencia.head(10)

,id_competencia,descricao,nome_competencia
0,1,Domínio básico de Python para scripts e automa...,Python Básico
1,2,SQL avançado para consultas complexas e otimiz...,SQL Avançado
2,3,Conhecimento intermediário de C# para desenvol...,C# Intermediário
3,4,Noções de arquitetura de software e design pat...,Arquitetura de Software
4,5,Experiência com metodologias ágeis como Scrum ...,Metodologias Ágeis
5,6,Conhecimento em React para desenvolvimento fro...,React
6,7,Análise de dados com ferramentas como Power BI...,Análise de Dados
7,8,Gestão de projetos com ferramentas como MS Pro...,Gestão de Projetos
8,9,Segurança da informação e boas práticas de pro...,Segurança da Informação
9,10,Comunicação eficaz e liderança de equipes técn...,Comunicação e Liderança


In [ ]:
df_competencia.isnull().sum()

id_competencia      0
descricao           0
nome_competencia    0
dtype: int64

In [ ]:
# Lista de prováveis níveis  possíveis
niveis = ['Básico', 'Intermediário', 'Avançado']

# Função para extrair o nível
def extrair_nivel(nome):
    for nivel in niveis:
        if nivel in nome:
            return nivel
    return None

# Criar a coluna 'nivel'
df_competencia['nivel'] = df_competencia['nome_competencia'].apply(extrair_nivel)

# Remover o nível do nome da competência
df_competencia['nome_competencia'] = df_competencia['nome_competencia'].str.replace(r'\s*(Básico|Intermediário|Avançado)', '', regex=True).str.strip()
df_competencia.head(10)

,id_competencia,descricao,nome_competencia,nivel
0,1,Domínio básico de Python para scripts e automa...,Python,Básico
1,2,SQL avançado para consultas complexas e otimiz...,SQL,Avançado
2,3,Conhecimento intermediário de C# para desenvol...,C#,Intermediário
3,4,Noções de arquitetura de software e design pat...,Arquitetura de Software,None
4,5,Experiência com metodologias ágeis como Scrum ...,Metodologias Ágeis,None
5,6,Conhecimento em React para desenvolvimento fro...,React,None
6,7,Análise de dados com ferramentas como Power BI...,Análise de Dados,None
7,8,Gestão de projetos com ferramentas como MS Pro...,Gestão de Projetos,None
8,9,Segurança da informação e boas práticas de pro...,Segurança da Informação,None
9,10,Comunicação eficaz e liderança de equipes técn...,Comunicação e Liderança,None


In [ ]:
# Dicionário de regras de classificação
regras_nivel = {
    "Noções de": "Básico",
    "Domínio básico de": "Básico",
    "Conhecimento intermediário de": "Intermediário",
    "Conhecimento em": "Intermediário", 
    "Conhecimento": "Intermediário", # caso geral
    "Experiência em" : "Avançado",
    "Experiência com": "Avançado",
    "avançado ": "Avançado"
}

# Função para definir o nível
def classificar_nivel(desc):
    for chave, nivel in regras_nivel.items():
        if chave.lower() in desc.lower():
            return nivel
    return None

# Criar nova coluna "nivel" baseada na descrição
df_competencia["nivel"] = df_competencia["descricao"].apply(classificar_nivel)

# Remover as palavras-chave da descrição
for chave in regras_nivel.keys():
    df_competencia["descricao"] = df_competencia["descricao"].str.replace(chave, "", case=False, regex=False)

# Limpar espaços extras
df_competencia["descricao"] = df_competencia["descricao"].str.strip()

print("Tabela Competencia:")
print(df_competencia)


In [ ]:
df_competencia["nivel"] = df_competencia["nivel"].fillna("Não informado")

In [ ]:
df_competencia.rename(columns=lambda col: "_".join(p.capitalize() for p in col.split("_")), inplace=True)

In [ ]:
df_competencia["Nivel_Competencia"]= df_competencia["Nivel"]
drop_columns = ["Nivel"]
df_competencia.drop(columns=drop_columns, inplace=True)
df_competencia.head(10)

,Id_Competencia,Descricao,Nome_Competencia,Nivel_Competencia
0,1,Domínio básico de Python para scripts e automa...,Python,Básico
1,2,SQL avançado para consultas complexas e otimiz...,SQL,Avançado
2,3,Conhecimento intermediário de C# para desenvol...,C#,Intermediário
3,4,Noções de arquitetura de software e design pat...,Arquitetura de Software,Não informado
4,5,Experiência com metodologias ágeis como Scrum ...,Metodologias Ágeis,Não informado
5,6,Conhecimento em React para desenvolvimento fro...,React,Não informado
6,7,Análise de dados com ferramentas como Power BI...,Análise de Dados,Não informado
7,8,Gestão de projetos com ferramentas como MS Pro...,Gestão de Projetos,Não informado
8,9,Segurança da informação e boas práticas de pro...,Segurança da Informação,Não informado
9,10,Comunicação eficaz e liderança de equipes técn...,Comunicação e Liderança,Não informado


In [ ]:
df_competencia["Descricao"]= df_competencia["Descricao"].str.capitalize()
df_competencia.head(10)

In [ ]:
df_competencia = df_competencia[df_competencia["Id_Competencia"] != 10]

In [ ]:
#df_competencia.to_csv("../ArquivosPostgresql/dados_tratados_postgresql_v2/competencia_postgres_tratado.csv", index=False)

In [ ]:
#variável de ambiente para construir o caminho
caminho_base_arquivos = os.environ.get("CAMINHO_ARQUIVOS_CSV")
caminho_saida = os.path.join(caminho_base_arquivos, "dados_tratados_v2", "competencia_postgres_tratado.csv")

#  variável caminho_saida
df_competencia.to_csv(caminho_saida, index=False)
